In [78]:
print(123)

123


In [79]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [80]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [81]:
v1 = model.encode(q1)

In [82]:
v1.shape

(384,)

In [83]:
v2 = model.encode(q2)

In [84]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [85]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [86]:
v1.dot(dv)

np.float32(0.32332402)

In [87]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [88]:
v2.dot(dv)

np.float32(0.019730477)

In [89]:
!curl -o ingest.py https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   738  100   738    0     0   1746      0 --:--:-- --:--:-- --:--:--  1757
100   738  100   738    0     0   1743      0 --:--:-- --:--:-- --:--:--  1752


In [90]:
from ingest import load_faq_data

documents = load_faq_data()

In [91]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [92]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [93]:
len(texts)

1368

In [94]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [95]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1368

In [96]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [97]:
import numpy as np
X = np.array(vectors)

In [98]:
scores = X.dot(v1)

In [99]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294094))

In [100]:
documents[553]

{'id': 'a9353fadfe',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."}

In [101]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [102]:
scores[top5]

array([0.76294094, 0.75793695, 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [103]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294094
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.75793695
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relat

In [104]:
top5

array([  2, 643, 925, 538,   7])

In [105]:
top5 = np.argsort(-scores)[:5]

In [106]:
top5

array([  2, 643, 925, 538,   7])

In [107]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [108]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the for

In [109]:
# !curl -o rag_helper.py https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py


In [110]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables
load_dotenv()

# Configure Groq
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)


In [111]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [112]:
import importlib
import rag_helper
    
importlib.reload(rag_helper)
    
from rag_helper import RAGBase
    

In [113]:


assistant = RAGBase(
    index=index,
    llm_client=client,
)

In [114]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)


"Yes, you can still sign up. You don't need to register for the course to confirm your spot. You're automatically accepted to the LLM Zoomcamp and can simply start learning and submitting homework."

In [115]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [116]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
)

In [117]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)


'Yes, you can still sign up for the course.'

In [118]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [125]:
vs_index.clear()

In [126]:
vs_index.fit(vectors, documents)


In [127]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [128]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [130]:
vs_index.close()